# Identifier-masked probe — the reviewer's first ask

**Runtime → GPU. Run all.** About 90 minutes.

The probe pools activations over the occurrence's own tokens, so it reads the
variable's *name*. The surface baseline is forbidden from reading it. That
asymmetry is disclosed in the paper but not controlled, and it is the single
biggest threat to the claim that the probe encodes roles rather than names.

This notebook removes it. `--pool context` averages the tokens **around** the
occurrence and excludes its own, so the forward pass is byte-identical to the
span-pooled run and only the read changes. Masking the name in the source would
also change the input distribution; this does not.

It produces the four-way comparison the review asks for, on the same
occurrences:

| | reads the name? | reads context? |
|---|---|---|
| probe, span-pooled | yes | no |
| **probe, context-pooled** | **no** | **yes** |
| surface n-gram, masked | no | yes |
| identifier alone | yes | no |

It also runs on the **three-way intersection** (869 problems shared by all
three languages) so every cell scores identical problem ids, and emits per-cell
predictions so the probe side gets problem-level confidence intervals.

In [ ]:
# 1 - setup
import pathlib, os
# The scripts this notebook needs (--pool context, --primary-feature,
# build_intersection, transfer_intervals) live here until the branch merges.
# Point at main once it has.
BRANCH = "review-hardening"
REPO = "/content/mech-interp"
if not pathlib.Path(REPO).exists():
    !git clone -q https://github.com/nolanlwin/mech-interp.git {REPO}
%cd {REPO}
!git fetch -q origin && git checkout -q -B {BRANCH} origin/{BRANCH} && git pull -q
!git log --oneline -1
!pip install -q transformers==5.8.0 torch numpy scikit-learn matplotlib tree_sitter \
  "tree-sitter-javascript>=0.25.0" "tree-sitter-php>=0.24.1"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
from google.colab import drive
# A mount left over from a previous session in the same runtime makes the plain
# call raise "mount failed". Retry once with force_remount, which clears it.
try:
    drive.mount("/content/drive")
except Exception as e:
    print(f"first mount attempt failed ({e}); retrying with force_remount")
    drive.mount("/content/drive", force_remount=True)
DEST = "/content/drive/MyDrive/mech-interp/masked"
!mkdir -p data/xlcost outputs/role_occ outputs/activations_xlcost outputs/crosslang {DEST}
!cp -rn {DEST}/stores/* outputs/activations_xlcost/ 2>/dev/null || true
!cp -n  {DEST}/role_occ/* outputs/role_occ/ 2>/dev/null || true
!cp -n  {DEST}/data_xlcost/* data/xlcost/ 2>/dev/null || true

# Existence is not the check. main carries extract_activations.py and
# baselines.py, but WITHOUT --pool and --primary-feature, so a file-exists
# assertion passes and cell 4 dies on an unrecognised argument. Assert the
# capability each cell actually uses.
import subprocess
REQUIRED = {
    "scripts/extract_activations.py": ["--pool", "--context-tokens"],
    "scripts/baselines.py":           ["--primary-feature"],
    "scripts/crosslang.py":           [],
    "scripts/role_occurrences.py":    [],
    "scripts/build_intersection.py":  [],
    "scripts/transfer_intervals.py":  ["--n-boot"],
}
problems = []
for script, flags in REQUIRED.items():
    if not pathlib.Path(script).exists():
        problems.append(f"{script} is missing")
        continue
    if not flags:
        continue
    helps = ""
    for argv in ([script, "--help"], [script, "run", "--help"],
                 [script, "transfer", "--help"]):
        r = subprocess.run(["python"] + argv, capture_output=True, text=True)
        helps += r.stdout + r.stderr
    absent = [f for f in flags if f not in helps]
    if absent:
        problems.append(f"{script} does not accept {absent}")
if problems:
    raise SystemExit(
        f"BRANCH={BRANCH!r} is not the right ref:\n  " + "\n  ".join(problems)
        + "\nPoint BRANCH at a branch carrying these, or merge it into main."
    )
print("all required scripts present, with the flags this notebook passes")
import torch
print(f"setup complete | cuda={torch.cuda.is_available()} "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''}")

In [ ]:
# 2 - CONFIG
MODEL  = "Qwen/Qwen2.5-Coder-1.5B"
ROLES  = ["accumulator", "iterator", "index_key"]
LANGS  = {"Python": "python", "Javascript": "javascript", "PHP": "php"}
SPLIT  = "train"
CONTEXT_TOKENS = 16          # tokens either side when pooling context only
PRIMARY = "window_masked"    # the pre-registered surface variant, not a per-cell max

import re as _re
slug = lambda mid: _re.sub(r"[^a-z0-9]", "", mid.split("/")[-1].lower())
SPAN_SLUG = slug(MODEL)
CTX_SLUG  = slug(f"{MODEL}#pool-context{CONTEXT_TOKENS}")
# The untrained floor must share the POOLING it is a floor for. A span-pooled
# untrained network is not the chance level for a context-pooled probe: the two
# read different token sets, so their achievable scores differ for reasons that
# have nothing to do with training.
RAND_CTX = f"{MODEL}#random-init-s0#pool-context{CONTEXT_TOKENS}"
RCTX_SLUG = slug(RAND_CTX)
print(f"{MODEL}\n  span-pooled store slug        : {SPAN_SLUG}"
      f"\n  context-pooled store slug     : {CTX_SLUG}"
      f"\n  untrained context-pooled slug : {RCTX_SLUG}")

In [ ]:
# 3 - corpora, occurrences, and the three-way intersection. CPU, ~15 min.
import json, itertools
for L, s in LANGS.items():
    if not pathlib.Path(f"data/xlcost/{s}_{SPLIT}.jsonl.stats.json").exists():
        !python scripts/xlcost_data.py build --language "{L}" --split {SPLIT} --out-dir data/xlcost
for L, s in LANGS.items():
    occ = f"outputs/role_occ/all_{s}_{SPLIT}.jsonl"
    if not pathlib.Path(occ + ".stats.json").exists():
        !python scripts/role_occurrences.py extract --input data/xlcost/{s}_{SPLIT}.jsonl --role all --output {occ}

!python scripts/build_intersection.py

absent = [f"outputs/role_occ/isect_{s}_{SPLIT}.jsonl" for s in LANGS.values()
          if not pathlib.Path(f"outputs/role_occ/isect_{s}_{SPLIT}.jsonl").exists()]
if absent:
    raise SystemExit(f"intersection build produced nothing for {absent}; read the output above.")
for s in LANGS.values():
    n = sum(1 for _ in open(f"outputs/role_occ/isect_{s}_{SPLIT}.jsonl"))
    print(f"  {s}: {n} occurrences on the shared problem set")

In [ ]:
# 4 - activations. GPU. Two stores per language: span-pooled and context-pooled.
#     ~90 min total. The context store is the identifier-masked condition.
for L, s in LANGS.items():
    for pool, sl in (("span", SPAN_SLUG), ("context", CTX_SLUG),
                     ("context-untrained", RCTX_SLUG)):
        out = f"outputs/activations_xlcost/isect_{s}_{SPLIT}_{sl}"
        if pathlib.Path(f"{out}/meta.json").exists():
            print(f"  have {out}")
            continue
        extra = ""
        if pool.startswith("context"):
            extra = f"--pool context --context-tokens {CONTEXT_TOKENS}"
        if pool == "context-untrained":
            extra += " --random-init --random-seed 0"
        !python scripts/extract_activations.py run \
          --canonical data/xlcost/{s}_{SPLIT}_isect.jsonl \
          --occurrences outputs/role_occ/isect_{s}_{SPLIT}.jsonl \
          --model-id {MODEL} --label-field role {extra} --out-dir {out}

bad = [f"outputs/activations_xlcost/isect_{s}_{SPLIT}_{sl}"
       for s in LANGS.values() for sl in (SPAN_SLUG, CTX_SLUG, RCTX_SLUG)
       if not pathlib.Path(f"outputs/activations_xlcost/isect_{s}_{SPLIT}_{sl}/meta.json").exists()]
if bad:
    raise SystemExit(f"no store written for {bad}; cell 5 has nothing to probe.")
print("\nall nine stores present")

# Persist the GPU work NOW, not at the end of the notebook. Everything after
# this point is cheap and interruptible; the extraction above is the only part
# that costs an hour, and a disconnect before the final cell used to throw it
# away. /content does not survive a reconnect. Drive does.
!mkdir -p {DEST}/stores {DEST}/role_occ {DEST}/data_xlcost
!cp -rn outputs/activations_xlcost/* {DEST}/stores/ 2>/dev/null || true
!cp -n outputs/role_occ/* {DEST}/role_occ/ 2>/dev/null || true
!cp -n data/xlcost/*_{SPLIT}*.jsonl* {DEST}/data_xlcost/ 2>/dev/null || true
print(f"stores saved to {DEST}; a disconnect from here costs minutes, not the GPU hour")

In [ ]:
# 5 - probe transfer, all three poolings, all six ordered pairs. GPU-light.
#     Safe to re-run: finished cells are skipped, so an interrupted run
#     continues where it stopped.
import itertools, glob, json

# Existence is not completion. A run killed mid-write leaves a truncated file,
# and skipping on existence would keep it forever and let cell 7 read it as a
# real result. Drop anything that does not parse or lacks the score.
incomplete = []
for f in glob.glob("outputs/crosslang/probe_*.json"):
    try:
        if "transfer_macro_f1_mean" not in json.loads(pathlib.Path(f).read_text()):
            incomplete.append(f)
    except Exception:
        incomplete.append(f)
for f in incomplete:
    pathlib.Path(f).unlink()
    print(f"  discarded incomplete {pathlib.Path(f).name}")
if incomplete:
    print(f"  {len(incomplete)} file(s) will be recomputed\n")

for sl, tag in ((SPAN_SLUG, "span"), (CTX_SLUG, "context"),
                (RCTX_SLUG, "context-untrained")):
    # The untrained condition is a floor, not an estimate to be compared
    # between cells, so it does not need the full seed sweep the trained
    # conditions get. Two seeds instead of five cuts it from roughly eighty
    # minutes to thirty without changing what it is used for.
    seeds = "--seeds 0 1" if sl == RCTX_SLUG else ""
    for role in ROLES:
        for a, b in itertools.permutations(LANGS.values(), 2):
            out = f"outputs/crosslang/probe_{role}_{a}_to_{b}_{sl}.json"
            if pathlib.Path(out).exists():
                continue
            !python scripts/crosslang.py run \
              --train-store outputs/activations_xlcost/isect_{a}_{SPLIT}_{sl} \
              --test-store  outputs/activations_xlcost/isect_{b}_{SPLIT}_{sl} \
              --role {role} {seeds} --output {out}
made = glob.glob("outputs/crosslang/probe_*.json")
expected = 3 * len(ROLES) * len(LANGS) * (len(LANGS) - 1)
if not made:
    raise SystemExit("no probe results written; read the output above.")
print(f"{len(made)} of {expected} probe result(s)")
if len(made) < expected:
    print("INCOMPLETE. Re-run this cell; it resumes. If it keeps stopping at the")
    print("same point, read the traceback above rather than re-running again.")

In [ ]:
# 6 - the model-free arm on the SAME occurrences, one pre-registered variant.
#     CPU, ~20 min. --primary-feature pins which family the emitted predictions
#     belong to, so the intervals in cell 7 describe one named estimator.
for role in ROLES:
    for a, b in itertools.permutations(LANGS.values(), 2):
        out = f"outputs/isect_wm/{role}_{a}_to_{b}.json"
        if pathlib.Path(out).exists():
            continue
        !python scripts/baselines.py transfer \
          --train-occurrences outputs/role_occ/isect_{a}_{SPLIT}.jsonl \
          --train-canonical   data/xlcost/{a}_{SPLIT}_isect.jsonl \
          --test-occurrences  outputs/role_occ/isect_{b}_{SPLIT}.jsonl \
          --test-canonical    data/xlcost/{b}_{SPLIT}_isect.jsonl \
          --label-field role --role {role} --matched \
          --primary-feature {PRIMARY} --output {out}
print(f"{len(glob.glob('outputs/isect_wm/*.json'))} baseline cell(s)")

In [ ]:
# 7 - problem-level confidence intervals, and the four-way comparison.
!python scripts/transfer_intervals.py --in outputs/isect_wm \
    --out results/lp4fm/transfer_intervals.csv --n-boot 2000

import csv, statistics as st, json
f = lambda r, k: float(r[k])
near = lambda rs: [r for r in rs if "python" not in (r["source"], r["target"])]
far  = lambda rs: [r for r in rs if "python" in (r["source"], r["target"])]
m    = lambda g, k: st.mean(f(r, k) for r in g)

def probe_rows(sl):
    out = []
    for p in glob.glob(f"outputs/crosslang/probe_*_{sl}.json"):
        d = json.loads(open(p).read())
        mm = _re.match(r"probe_([a-z_]+)_([a-z]+)_to_([a-z]+)_", pathlib.Path(p).name)
        if mm:
            out.append({"role": mm.group(1), "source": mm.group(2), "target": mm.group(3),
                        "v": d["transfer_macro_f1_mean"]})
    return out

print(f"\n{'condition':<34}{'close':>9}{'Python':>9}{'effect':>9}")
base = list(csv.DictReader(open("results/lp4fm/transfer_intervals.csv")))
base = [r for r in base if r["role"] != "ALL"]
print(f"{'surface n-gram (' + PRIMARY + ')':<34}{m(near(base),'macro_f1'):>9.3f}"
      f"{m(far(base),'macro_f1'):>9.3f}{m(far(base),'macro_f1')-m(near(base),'macro_f1'):>+9.3f}")
for sl, tag in ((SPAN_SLUG,  "probe, span-pooled (reads name)"),
                (CTX_SLUG,   "probe, context-pooled (no name)"),
                (RCTX_SLUG,  "UNTRAINED, context-pooled (the floor)")):
    rs = probe_rows(sl)
    if rs:
        print(f"{tag:<34}{m(near(rs),'v'):>9.3f}{m(far(rs),'v'):>9.3f}"
              f"{m(far(rs),'v')-m(near(rs),'v'):>+9.3f}")

print("\nThe question is the gap between the last two rows. A context-pooled probe")
print("must be read against a context-pooled floor: the span-pooled untrained")
print("network is not the chance level for a different token set. If the trained")
print("and untrained context-pooled rows are close, then with the identifier")
print("removed the probe reads little the model learned, and the paper's central")
print("claim does not survive in its current form.")
# The paired test the review asks for: surface and probe on the SAME
# occurrences, one problem resample applied to both, so the difference between
# them gets an interval rather than two point estimates side by side.
import sys
sys.path.insert(0, "scripts")
from bootstrap_ci import paired_delta_ci
import numpy as np

def paired(role, a, b, probe_slug):
    pj = pathlib.Path(f"outputs/crosslang/probe_{role}_{a}_to_{b}_{probe_slug}.json")
    bj = pathlib.Path(f"outputs/isect_wm/{role}_{a}_to_{b}.json")
    if not (pj.exists() and bj.exists()):
        return None
    P, B = json.loads(pj.read_text()), json.loads(bj.read_text())
    key = lambda rows: {r["occurrence_id"]: r for r in rows
                        if r.get("seed") in (0, None) and r.get("occurrence_id")}
    kp, kb = key(P.get("test_predictions") or []), key(B.get("test_predictions") or [])
    both = sorted(set(kp) & set(kb))
    if len(both) < 50:
        return None
    y  = np.array([kp[o]["y_true"] for o in both])
    pa = np.array([kp[o]["y_pred"] for o in both])
    pb = np.array([kb[o]["y_pred"] for o in both])
    cl = np.array([kb[o]["cluster"] for o in both])
    return paired_delta_ci(y, pa, pb, cl, sorted(set(y.tolist())), n_boot=1000)

print(f"\n{'cell':<34}{'probe - surface':>18}{'95% CI':>22}")
for role in ROLES:
    for a, b in itertools.permutations(LANGS.values(), 2):
        r = paired(role, a, b, CTX_SLUG)
        if r:
            # paired_delta_ci names the statistic "delta", not "point".
            print(f"  {role} {a[:4]}->{b[:4]:<16}{r['delta']:>+16.3f}"
                  f"   [{r['ci_low']:+.3f}, {r['ci_high']:+.3f}]"
                  f"{'  *' if r.get('excludes_zero') else ''}")
print("\nCells marked * have an interval excluding zero: the two methods differ")
print("on the same occurrences, not merely on average across cells.")


In [ ]:
# 8 - save to Drive. Push results only; nothing here is anonymised.
!mkdir -p {DEST}/stores {DEST}/role_occ {DEST}/data_xlcost {DEST}/crosslang {DEST}/isect_wm
!cp -r outputs/activations_xlcost/* {DEST}/stores/ 2>/dev/null || true
!cp outputs/role_occ/* {DEST}/role_occ/ 2>/dev/null || true
!cp data/xlcost/*_{SPLIT}*.jsonl* {DEST}/data_xlcost/ 2>/dev/null || true
!cp outputs/crosslang/*.json {DEST}/crosslang/ 2>/dev/null || true
!cp outputs/isect_wm/*.json {DEST}/isect_wm/ 2>/dev/null || true
print(f"artifacts saved to {DEST}")
print("\nPaste cell 7's table back into the chat; the paper's numbers follow from it.")